## Feature Engineering Goals
| Feature                    | Description                                   | Calculation Source           |
| -------------------------- | --------------------------------------------- | ---------------------------- |
| `Total_Payments`           | Sum of all payments per loan                  | fact_payments                |
| `Avg_Payment_Amount`       | Average payment per loan                      | fact_payments                |
| `Payment_Count`            | Count of payments per loan                    | fact_payments                |
| `Total_Days_Late`          | Sum of all delay days                         | fact_payments                |
| `Late_Payment_Rate`        | % of payments that were late                  | fact_payments                |
| `Latest_Remaining_Balance` | Last known remaining balance                  | fact_payments                |
| `Is_Default`               | Flag based on Default_Status or Late Payments | loan_applications + payments |
| `Customer_Age_Group`       | Categorize customer’s age                     | loan_applications.Age        |
| `Income_Category`          | Categorize income                             | dim_customers.Annual_Income  |



### Connect & Loan Data

In [1]:
import pyodbc
import pandas as pd 
import numpy as np

DRIVER_NAME = 'ODBC Driver 17 for SQL Server'
SERVER_NAME = r'DESKTOP-L3GBMQ5\SQLEXPRESS'
DATABASE_NAME = 'Financial_CaseStudy'

connection_string = (
    f"DRIVER={{{DRIVER_NAME}}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)
cursor = conn.cursor()
print("Connected successfully!")

Connected successfully!


In [2]:
df_customers = pd.read_sql("SELECT * FROM stg.stg_dim_customers", conn)
df_loans = pd.read_sql("SELECT * FROM stg.stg_loan_applications", conn)
df_payments = pd.read_sql("SELECT * FROM stg.stg_fact_payments", conn)

C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_4372\2238739616.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customers = pd.read_sql("SELECT * FROM stg.stg_dim_customers", conn)
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_4372\2238739616.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_loans = pd.read_sql("SELECT * FROM stg.stg_loan_applications", conn)
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_4372\2238739616.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_payments = pd.read_sql("SELECT * FROM

In [18]:
df_customers

,Customer_ID,Full_Name,Gender,Date_of_Birth,Region,Education_Level,Employment_Status,Annual_Income,Credit_History_Length,Credit_Score
0,1001,Customer_1,Male,1996-07-22,North,High School,Employed,102988.0,3,808
1,1002,Customer_2,Female,2012-01-23,North,Bachelor,Retired,83342.0,1,780
2,1003,Customer_3,Male,1986-08-12,West,Master,Self-employed,139516.0,4,413
3,1004,Customer_4,Male,1996-05-18,West,Bachelor,Employed,43951.0,20,660
4,1005,Customer_5,Male,1997-11-21,West,High School,Employed,26307.0,2,640
...,...,...,...,...,...,...,...,...,...,...
495,1496,Customer_496,Male,1988-12-04,North,Master,Self-employed,72832.0,18,559
496,1497,Customer_497,Male,2000-07-24,Central,Bachelor,Employed,53151.0,8,401
497,1498,Customer_498,Female,1989-09-08,West,Bachelor,Employed,104303.0,22,805
498,1499,Customer_499,Male,2013-08-17,West,Master,Retired,36308.0,24,546


In [19]:
df_loans

,Loan_ID,Customer_ID,Loan_Amount,Loan_Term_Months,Interest_Rate,Loan_Purpose,Application_Date,Approval_Status,Default_Status,Monthly_Income,Credit_Score,Employment_Length,Marital_Status,Age,Region,Gender
0,5001,1059,48370.0,24,11.91,Education,2024-11-11,Approved,Paid,10828.833008,391,18,Married,24,North,Male
1,5002,1132,43396.0,36,10.15,Car,2023-01-05,Rejected,None,2990.500000,696,19,Divorced,39,East,Male
2,5003,1104,13328.0,24,12.97,Business,2022-07-10,Approved,Paid,2570.416748,754,3,Single,9,Central,Female
3,5004,1195,20534.0,36,15.47,Education,2022-12-13,Approved,Paid,10096.500000,677,9,Single,45,West,Male
4,5005,1413,24744.0,12,8.90,Business,2022-09-10,Approved,Paid,6738.333496,789,4,Married,5,Central,Female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,5996,1147,6881.0,36,6.81,Business,2022-09-28,Approved,Paid,3032.500000,499,6,Divorced,37,Central,Male
996,5997,1118,44567.0,36,11.76,Business,2021-05-30,Approved,Defaulted,8870.166992,637,13,Married,5,South,Female
997,5998,1399,18675.0,12,5.48,Personal,2021-08-03,Approved,Paid,7271.750000,441,7,Single,31,South,Female
998,5999,1382,9586.0,24,6.05,Car,2022-12-17,Approved,Defaulted,11057.666992,699,5,Divorced,10,North,Female


In [20]:
df_payments

,Payment_ID,Loan_ID,Payment_Date,Payment_Amount,Remaining_Balance,Is_Late,Days_Late
0,8001,5447,2021-05-01,645.760010,26237.949219,False,0
1,8002,5785,2021-12-11,743.880005,4793.629883,True,12
2,8003,5497,2020-10-08,1407.630005,35170.910156,False,0
3,8004,5561,2020-07-25,1383.979980,31168.130859,False,0
4,8005,5484,2023-10-12,1130.380005,7629.930176,False,0
...,...,...,...,...,...,...,...
2995,10996,5650,2021-08-21,1355.359985,39588.929688,False,0
2996,10997,5983,2022-11-10,803.679993,19366.570312,False,0
2997,10998,5466,2022-08-08,870.960022,18723.259766,False,0
2998,10999,5288,2023-07-05,632.590027,25798.550781,True,18


### SQL Query - Aggregate Payments

In [22]:
df_payment_summary = (
    df_payments.groupby("Loan_ID")
    .agg(
        Total_Payments=('Payment_Amount', 'sum'),
        Payment_Count=('Payment_ID', 'count'),
        Avg_Payment_Amount=('Payment_Amount', 'mean'),
        Total_Days_Late=('Days_Late', 'sum'),
        Late_Payment_Rate=('Is_Late', 'mean'),
        Latest_Remaining_Balance=('Remaining_Balance', 'last')
    )
    .reset_index()
)

### Merge Loan & Payment Data

In [23]:
if 'Loan_ID' in df_loans.columns:
    df_loan_features = pd.merge(df_loans, df_payment_summary, on='Loan_ID', how='left')
else:
    print('Your loan_applications table has no Loan_ID. Please confirm this first exists to join with payments.')
    df_loan_features = df_loans.copy()

In [24]:
# Fill NaN values for loans without payments
numeric_cols = ['Total_Payments', 'Payment_Count', 'Avg_Payment_Amount',
                'Total_Days_Late', 'Late_Payment_Rate', 'Latest_Remaining_Balance']
for col in numeric_cols:
    df_loan_features[col] = df_loan_features[col].fillna(0)

### Derived Loan Performance Indicators

In [25]:
df_loan_features['Is_Default'] = df_loan_features['Default_Status'].apply(
    lambda x: 1 if str(x).lower() in ['yes', 'y','1','true'] else 0
)

### Merge Customer Demographics

In [26]:
if 'Customer_ID' in df_loan_features.columns:
    df_final = pd.merge(df_loan_features, df_customers, on='Customer_ID', how='left')
else:
    df_final = df_loan_features.copy()

### Categorize Age and Income

In [27]:
if 'Age' in df_final.columns:
    bins_age = [0, 25, 35, 50, 65, 100]
    labels_age = ['<25', '25-35', '36-50', '51-65', '65+']
    df_final['Customer_Age_Group'] = pd.cut(df_final['Age'], bins=bins_age, labels=labels_age, right=False)

In [28]:
if 'Annual_Income' in df_final.columns:
    bing_income = [0, 3000, 7000, 12000, 20000, 100000]
    labels_income = ['Low', 'Lower-Mid', 'Mid', 'Upper-Mid', 'High']
    df_final['Income_Level'] = pd.cut(df_final['Annual_Income'], bins=bing_income, labels=labels_income, right=False)

In [29]:
df_final

,Loan_ID,Customer_ID,Loan_Amount,Loan_Term_Months,Interest_Rate,Loan_Purpose,Application_Date,Approval_Status,Default_Status,Monthly_Income,...,Gender_y,Date_of_Birth,Region_y,Education_Level,Employment_Status,Annual_Income,Credit_History_Length,Credit_Score_y,Customer_Age_Group,Income_Level
0,5001,1059,48370.0,24,11.91,Education,2024-11-11,Approved,Paid,10828.833008,...,Male,2001-06-03,North,Master,Employed,129946.0,6,391,<25,NaN
1,5002,1132,43396.0,36,10.15,Car,2023-01-05,Rejected,None,2990.500000,...,Male,1985-10-31,East,Bachelor,Self-employed,35886.0,8,696,36-50,High
2,5003,1104,13328.0,24,12.97,Business,2022-07-10,Approved,Paid,2570.416748,...,Female,2016-09-14,Central,High School,Unemployed,30845.0,8,754,<25,High
3,5004,1195,20534.0,36,15.47,Education,2022-12-13,Approved,Paid,10096.500000,...,Male,1980-04-14,West,Master,Employed,121158.0,22,677,36-50,NaN
4,5005,1413,24744.0,12,8.90,Business,2022-09-10,Approved,Paid,6738.333496,...,Female,2019-11-01,Central,Bachelor,Employed,80860.0,16,789,<25,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,5996,1147,6881.0,36,6.81,Business,2022-09-28,Approved,Paid,3032.500000,...,Male,1988-01-25,Central,Master,Employed,36390.0,8,499,36-50,High
996,5997,1118,44567.0,36,11.76,Business,2021-05-30,Approved,Defaulted,8870.166992,...,Female,2020-07-04,South,Master,Employed,106442.0,20,637,<25,NaN
997,5998,1399,18675.0,12,5.48,Personal,2021-08-03,Approved,Paid,7271.750000,...,Female,1994-09-08,South,Master,Unemployed,87261.0,20,441,25-35,High
998,5999,1382,9586.0,24,6.05,Car,2022-12-17,Approved,Defaulted,11057.666992,...,Female,2015-07-12,North,Master,Employed,132692.0,11,699,<25,NaN


### Save to SQL Server

In [30]:
cursor = conn.cursor()
cursor.execute("IF OBJECT_ID('dw_loan_analysis') IS NOT NULL DROP TABLE dw_loan_analysis;")
conn.commit()

In [36]:
from sqlalchemy import create_engine

# Your local SQL Server details
server = r"DESKTOP-L3GBMQ5\SQLEXPRESS"   # note the raw string 'r' for backslash
database = "Financial_CaseStudy"

# ✅ Correct connection string (no ".0")
engine = create_engine(
    f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# Push df_final to SQL Server
df_final.to_sql('dw_loan_analysis', con=engine, if_exists='replace', index=False)

print("\n💾 Data successfully saved as 'stg_loan_analysis' in SQL Server!")



💾 Data successfully saved as 'stg_loan_analysis' in SQL Server!
